# HOMER × Pagani 2026 — per-model subtype translation (corrected)

How HOMER's optimal-transport coupling π translates Pagani's autism connectivity **subtypes**
from mouse to human, and places each of the 20 mouse models on the hyper↔hypo axis.

> **This notebook was rewritten on 2026-06-10** after the Gozzi lab shared the clean data package.
> It supersedes the earlier *exploratory* version, which PCA/KMeans-clustered the 20 models in the
> Excel-corrupted 1,491-feature space and labelled subtypes from a biological prior that was
> **inverted**. Both problems are fixed here. Full ingest/validation:
> `experiments/pagani_2026_per_model/DATA_VALIDATION_2026-06-10.md`.

**Two corrections that drive everything below:**
1. The 1,491 Fig 1c features are *not* decodable to voxels (the shared wo-cerebellum mask has
   10,111 voxels, not 1,491; there is no published feature→voxel key). So we do **not** attempt a
   per-voxel decode.
2. The subtype split is read from the clean CSV's row order and **verified from the data**
   (mean global connectivity > 0 for hyper, < 0 for hypo): rows 1–9 hyper (n=9), rows 10–20 hypo (n=11).

## Setup

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
from importlib import import_module
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'experiments' / 'pagani_2026_per_model'))

# The corrected experiment module is the single source of truth for the logic.
pm = import_module('01_per_model_clustering')
pi = np.load(ROOT / 'outputs/coupling/pi_fc_plus_SC_with_all_packs.npy')   # current production π
print(f'HOMER π: {pi.shape}, total mass {pi.sum():.3f}')

## Step 1 — Clean Fig 1c and the verified subtype split

The clean `sorted_etiology_by_feature_matrix.csv` is the de-corrupted MOESM6 Fig 1c
(20 models × 1,491 voxelwise weighted-degree-centrality features). We assign subtypes by
row order and **assert** the split against mean-connectivity sign — so the labels are earned,
not guessed.

In [ ]:
X, labels = pm.load_clean_figura_1c()
subtype = pm.derive_and_verify_subtypes(X, labels)   # raises if the sign check fails
df = pd.DataFrame({'model': labels,
                   'mean_connectivity': X.mean(1).round(3),
                   'subtype': subtype})
print(f'verified split: {subtype.count("hyper")} hyper / {subtype.count("hypo")} hypo')
df

**Note the correction.** The old version of this notebook labelled Fmr1/Chd8/Tsc2 as *hypo* and
16p11.2/Sgsh/Ube3a as *hyper* — exactly backwards. The data show Fmr1/Chd8/Tsc2/Il6/Cdkl5[ko]
are **hyper** (positive mean connectivity) and the synaptic models (Shank3, En2, …) are **hypo**,
matching the paper's n=9/n=11 split.

## Step 2 — Per-model membership on the hyper↔hypo axis (leave-one-out)

Each model is correlated to the *mean* hyper and hypo feature signature, excluding itself,
so its placement isn't circular. 17/20 fall on their own subtype side; the exceptions are the
near-zero-polarization models.

In [ ]:
members = pm.loo_membership(X, subtype)
n_ok = sum(members[i]['predicted_side'] == subtype[i] for i in range(len(labels)))
print(f'leave-one-out consistency: {n_ok}/{len(labels)}')
mdf = pd.DataFrame([{'model': labels[i], 'subtype': subtype[i],
                     'membership_score': round(members[i]['membership_score'], 3),
                     'predicted_side': members[i]['predicted_side']}
                    for i in range(len(labels))]).sort_values('membership_score')
mdf.reset_index(drop=True)

## Step 3 — Subtype translation through π

Each subtype's mouse network signature (Pagani ED Fig 1) is routed through π to human-parcel
space and aggregated to Pagani's 8 human networks, then compared to the *observed* human subtype
pattern (Fig 4e). This uses Pagani's own published network matrices and does not depend on the
1,491-feature decode.

In [ ]:
trans = pm.subtype_translation_through_pi()
xc = trans['cross_correlation']
print('cross-species correlation (predicted human ↔ observed human):')
print(f"  pred_hyper · obs_hyper = {xc['pred_hyper__obs_hyper']:+.3f}  (vs obs_hypo {xc['pred_hyper__obs_hypo']:+.3f})")
print(f"  pred_hypo  · obs_hypo  = {xc['pred_hypo__obs_hypo']:+.3f}  (vs obs_hyper {xc['pred_hypo__obs_hyper']:+.3f})")
print(f"  subtype-specific — hyper: {trans['subtype_specific_hyper']}, hypo: {trans['subtype_specific_hypo']}")

**Result — apparent "hyper-specific" pattern, but this is NOT an inferential result.** The mouse *hyper* signature correlates with the human *hyper* pattern (+0.35) more than human hypo (−0.25), while the *hypo* signature does not match human hypo (−0.13). It is tempting to read this as "HOMER recovers the hyper subtype but not hypo."

> **⚠️ Do not headline this.** The "subtype-specific" flag is just `r(pred_hyper,obs_hyper) > r(pred_hyper,obs_hypo)` — **not** a significance test. A **permuted-π null reproduces "hyper-specific" in ~100% of trials**, and the observed r = +0.35 sits *below* the null mean. This is the confounded **"Test 2a" absolute-correlation approach**: it is forced by the magnitude structure of the observed human maps (hyper network intensities are large, 60–244; hypo are tiny, 0.5–6), so any non-negative routing correlates with hyper and not hypo *by construction*. Treat this as illustrative of the pipeline, **not** evidence of cross-species hyper translation. The valid, magnitude-cancelling test is the contrast-based **Test 2c** (notebook 05 / `../autism_subtypes/`), which is r = +0.55 but **n.s. under a fair spin null** (p = 0.19). (An earlier run on the base `pi_fc_plus_SC.npy` mis-reported both subtypes as specific — a wrong-π artifact; see `_audit/FINDINGS_LOG.md` F-001/F-005.)

In [ ]:
fig = ROOT / 'outputs/figures/pagani_subtype_translation_corrected.png'
if fig.exists():
    display(Image(str(fig)))
else:
    print('Run experiments/pagani_2026_per_model/02_plot.py to generate the figure.')

## Step 4 — Parcel-resolution spatial routing (Direction 1) — SUPERSEDED

> **⚠️ Superseded (2026-06-11), kept for the honest negative result.** This routing predicts the human subtype Δ-*matrix* (a continuous-map correlation — HOMER's weak mode, **n.s. under a fair spin null**) and aggregates over all 13 conserved regions uniformly. Neither matches what Pagani actually do: they use only the 5 hypo-prominent / 3 hyper-prominent regions, and their human step is a discrete *classification*, not a Δ-matrix correlation. The corrected, paper-faithful analysis is **Step 5** below (`04`/`05`/`06`).

Driving the mouse side from the **Fig 1d occurrence maps** (aggregated to the 13 conserved regions via a verified Allen region-name bridge) instead of the 9-network matrices. Run `experiments/pagani_2026_per_model/03_spatial_subtype_routing.py` to (re)generate the log loaded below.

In [ ]:
sp_path = ROOT / 'outputs/logs/pagani_spatial_subtype_routing.json'
if sp_path.exists():
    sp = json.loads(sp_path.read_text())
    x = sp['cross_correlation']
    print(f"matched parcels: {sp['n_matched_parcels']}/1864")
    print(f"pred_hyper · obs_hyper = {x['pred_hyper__obs_hyper']:+.3f}  (vs obs_hypo {x['pred_hyper__obs_hypo']:+.3f})")
    print(f"pred_hypo  · obs_hypo  = {x['pred_hypo__obs_hypo']:+.3f}  (vs obs_hyper {x['pred_hypo__obs_hyper']:+.3f})")
else:
    print('Run 03_spatial_subtype_routing.py first.')

**Finding — a hyper>hypo asymmetry, but neither direction is inferential.** At parcel resolution the *hyper* contrast is positive and *hypo* is not, echoing Step 3's asymmetry. But (as in Step 3) this is **not** a significance test: the contrast p-value here also rides a strongly negative permuted-π null, and the routing tests a continuous-map correlation that does **not** survive a fair spin null. One concrete mechanism for the asymmetry: the occurrence maps are **unsigned consistency counts** (0–5), so they cannot carry the hypo subtype's connectivity *direction*. (⚠️ An earlier version of this cell claimed "the signed network matrices in Step 3 *do* recover both subtypes" — that was wrong; Step 3 recovers hyper, **not** hypo.) A clean, *signed* parcel-resolution test needs the per-model degree-centrality NIfTIs requested in `experiments/pagani_2026_per_model/email_draft_per_model_nifti.md`.

## Step 5 — The corrected, paper-faithful analysis (04 / 05 / 06)

Re-reading Pagani's Methods shows the human step is a discrete **classification** built from a mouse→human **name-match**, not a Δ-matrix correlation. That name-match is exactly what HOMER's π replaces — and it is HOMER's *validated* discrete mode. These three scripts implement the corrected analysis; the headline numbers are loaded below (full write-up in `../autism_subtypes/abide_subtype/README.md`).

In [ ]:
masks = json.loads((ROOT / 'outputs/logs/pagani_homer_human_masks.json').read_text())
sub   = json.loads((ROOT / 'outputs/logs/abide_homer_subtyping.json').read_text())
cont  = json.loads((ROOT / 'outputs/logs/abide_continuous_subtype.json').read_text())
h, n = sub['results']['homer'], sub['results']['name']

print(f"04  π-derived human masks vs name-match: argmax agrees {masks['argmax_agree']}/{masks['n_regions']} prominent regions")
print(f"05  ABIDE re-subtyping: HOMER {h['pct_subtyped']}%  vs  name {n['pct_subtyped']}%  "
      f"(Pagani ~{sub['pagani_reference_pct']:.0f}%), label agreement {sub['label_agreement']*100:.0f}%")
av = cont['asd_vs_ctrl']
worst = max(cont['ados_doseresponse'].values(), key=lambda r: abs(r['spearman_rho']))
print(f"06  continuous hyper↔hypo axis: ASD vs ctrl p={av['mannwhitney_p']:.2f}; "
      f"strongest ADOS |ρ|={abs(worst['spearman_rho']):.2f} (n.s.) — no severity dose-response")

**Interpretation.** With the name-match replaced by HOMER's learned coupling, the discrete subtyping **reproduces Pagani's** (HOMER 21.3% vs name-matched 22.3%, 93% label agreement) — independent validation, since π never saw their data. Removing the hard threshold to score the whole continuum (06) is a **clean negative** (no ASD/control shift, no ADOS dose-response). Together this is the same dichotomy seen throughout: **discrete cross-species correspondence is real and survives fair nulls; continuous/graded translation does not.**

## Summary

- **Data validated**, 1,491-feature voxel decode debunked, subtype labels corrected (were inverted).
- **Per-model membership** places all 20 models on the hyper↔hypo axis (17/20 leave-one-out consistent).
- **Subtype translation through π (Step 3)** shows an apparent hyper-specificity, but it is **not inferential** (a permuted-π null reproduces it ~100% of trials; the magnitude-cancelling Test 2c is n.s. under a fair spin null). Treat as illustrative.
- **Parcel-resolution routing (Step 4) is SUPERSEDED** — a continuous-map correlation that fails a fair null and isn't Pagani's procedure.
- **The corrected, paper-faithful analysis (Step 5)** is the real result: HOMER's π re-subtypes ABIDE just like Pagani's name-match (93% agreement) — a **discrete** success — while the **continuous** severity axis is a clean null. Same dichotomy as the rest of HOMER.

_Related: notebook `05_pagani_2026_validation.ipynb` covers the broader Test 1–4 cross-species validation and the synthesis table._